# 01 - Data Preparation & Chunked ETL Pipeline (Google Colab)

### Customer Support on Twitter Dataset Preprocessing
**Project:** Customer Support RAG Chatbot  
**Execution Environment:** Google Colab (Free CPU / T4 GPU)  
**Storage Backend:** Google Drive (`/content/drive/MyDrive/chatbot_data/`)

---
### Objectives
1. Mount Google Drive for persistent storage of raw, cleaned, and tokenized datasets.
2. Download the Kaggle **Customer Support on Twitter** (`thoughtvector/customer-support-on-twitter`) dataset.
3. Stream the ~1GB `twcs.csv` in chunks (`chunksize=50,000`) to completely eliminate Out-Of-Memory (OOM) crashes on Colab's 12GB RAM limit.
4. Filter high-value customer inquiries and verified brand responses (@AppleSupport, @AmazonHelp, @Uber_Support, @SpotifyCares, @Delta).
5. Clean, anonymize, and pair inquiries with brand solutions into standard JSONL format.
6. Export `cleaned_customer_support_sample.jsonl` (50,000 high-quality QA pairs) to Google Drive.



In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

DRIVE_BASE_DIR = "/content/drive/MyDrive/chatbot_data"
os.makedirs(f"{DRIVE_BASE_DIR}/raw", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/processed", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/tokenized", exist_ok=True)

print(f"[OK] Google Drive mounted successfully.")
print(f"[OK] Workspace created at: {DRIVE_BASE_DIR}")



In [ ]:
# Step 2: Install required lightweight utilities
!pip install -q kaggle pandas tqdm



### Step 3: Kaggle Authentication & Dataset Download
Provide your `kaggle.json` credentials either by placing them in `/content/drive/MyDrive/kaggle.json` or by setting Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`).



In [ ]:
import json
import shutil
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

# Check if kaggle.json is in Drive
drive_kaggle = Path("/content/drive/MyDrive/kaggle.json")
if drive_kaggle.exists():
    shutil.copy(drive_kaggle, kaggle_json)
    kaggle_json.chmod(0o600)
    print("[OK] Loaded kaggle.json from Google Drive.")
else:
    # Check Colab userdata secrets
    try:
        from google.colab import userdata
        k_user = userdata.get('KAGGLE_USERNAME')
        k_key = userdata.get('KAGGLE_KEY')
        with open(kaggle_json, 'w') as f:
            json.dump({"username": k_user, "key": k_key}, f)
        kaggle_json.chmod(0o600)
        print("[OK] Configured kaggle credentials from Colab Secrets.")
    except Exception as e:
        print("[!] No automatic Kaggle credentials found.")
        print("[!] Please upload kaggle.json or provide manually to download twcs.csv.")

# Download dataset if not already present
raw_csv_path = Path("/content/twcs.csv")
drive_csv_path = Path(f"{DRIVE_BASE_DIR}/raw/twcs.csv")

if drive_csv_path.exists():
    print(f"[OK] Found twcs.csv in Drive: {drive_csv_path}")
    raw_csv_path = drive_csv_path
elif raw_csv_path.exists():
    print(f"[OK] Found local twcs.csv in /content")
else:
    print("[*] Downloading dataset from Kaggle...")
    !kaggle datasets download -d thoughtvector/customer-support-on-twitter -p /content --unzip
    if raw_csv_path.exists():
        print("[*] Backing up raw twcs.csv to Google Drive for future sessions...")
        shutil.copy(raw_csv_path, drive_csv_path)
        print("[OK] Backup complete.")



### Step 4: Chunked ETL Pipeline (OOM Prevention)
The `twcs.csv` file contains nearly 3 million tweets (~1GB). Loading the entire dataset into pandas on a 12GB RAM Colab environment causes kernel crashes.
We process the file in streaming chunks of `50,000` rows, filtering specifically for:
- Top reputable brands: `@AppleSupport`, `@AmazonHelp`, `@Uber_Support`, `@SpotifyCares`, `@Delta`, `@AmericanAir`, `@NikeSupport`.
- Inbound inquiries from customers and outbound authoritative replies.



In [ ]:
import pandas as pd
import re
from tqdm.auto import tqdm

TARGET_BRANDS = {
    "AppleSupport", "AmazonHelp", "Uber_Support", 
    "SpotifyCares", "Delta", "AmericanAir", "NikeSupport"
}

def clean_tweet_text(text: str) -> str:
    """Cleans raw tweet text by removing URLs and standardizing whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Anonymize random numeric customer IDs (e.g. @115858 -> @customer)
    text = re.sub(r'@\d+', '@customer', text)
    # Standardize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("[*] Starting chunked scan to build tweet index...")
CHUNK_SIZE = 50_000

# Store tweet mappings: tweet_id -> {text, author, inbound, response_to, brand}
inbound_tweets = {}
brand_responses = []

for chunk in tqdm(pd.read_csv(raw_csv_path, chunksize=CHUNK_SIZE, low_memory=False)):
    # Filter for target brands or responses to/from them
    for _, row in chunk.iterrows():
        t_id = row['tweet_id']
        author = str(row['author_id'])
        inbound = bool(row['inbound'])
        text = str(row['text'])
        response_to = row['in_response_to_tweet_id']
        
        if inbound:
            # Store customer inquiries that might be answered
            inbound_tweets[t_id] = {
                "text": clean_tweet_text(text),
                "author": author
            }
        else:
            # Check if author is one of our target brands
            if author in TARGET_BRANDS and pd.notna(response_to):
                brand_responses.append({
                    "response_id": t_id,
                    "in_response_to_tweet_id": int(response_to),
                    "brand": author,
                    "response_text": clean_tweet_text(text)
                })

print(f"[OK] Inbound tweets indexed: {len(inbound_tweets):,}")
print(f"[OK] Brand responses collected: {len(brand_responses):,}")



### Step 5: Pairing Inquiries with Responses & JSONL Generation
We match inbound questions with brand answers to build high-quality `(instruction, response)` conversational pairs.



In [ ]:
import json
import random

qa_pairs = []

for resp in tqdm(brand_responses, desc="Matching QA pairs"):
    parent_id = resp["in_response_to_tweet_id"]
    if parent_id in inbound_tweets:
        inquiry = inbound_tweets[parent_id]["text"]
        answer = resp["response_text"]
        brand = resp["brand"]
        
        # Quality filters: minimum length, non-empty, avoid pure redirect bots
        if len(inquiry) >= 20 and len(answer) >= 25:
            if not answer.lower().startswith("please dm us") or len(answer) > 60:
                qa_pairs.append({
                    "instruction": inquiry,
                    "response": answer,
                    "brand": brand,
                    "metadata": {
                        "inquiry_id": parent_id,
                        "response_id": resp["response_id"]
                    }
                })

print(f"[OK] Total high-quality matched pairs: {len(qa_pairs):,}")

# Shuffle and sample 50,000 pairs for balanced training and indexing
random.seed(42)
random.shuffle(qa_pairs)
selected_pairs = qa_pairs[:50_000]

output_jsonl_path = f"{DRIVE_BASE_DIR}/processed/cleaned_customer_support_sample.jsonl"
with open(output_jsonl_path, "w", encoding="utf-8") as f:
    for pair in selected_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"[OK] Successfully exported {len(selected_pairs):,} pairs to:")
print(f"     {output_jsonl_path}")



### Step 6: Dataset Summary & Sanity Check
Inspect sample entries and brand distributions.



In [ ]:
from collections import Counter

brand_counts = Counter(p["brand"] for p in selected_pairs)
print("Brand distribution in final 50,000 sample:")
for brand, count in brand_counts.most_common():
    print(f"  - {brand:18s}: {count:,} samples ({count/len(selected_pairs)*100:.1f}%)")

print("\nSample Entry Preview:")
print(json.dumps(selected_pairs[0], indent=2))

